In [4]:
import numpy as np
import gymnasium as gym
import random

# Initialize environment
env = gym.make("Taxi-v3", render_mode="rgb_array")

# Initialize Q-table
Qtable = np.zeros((env.observation_space.n, env.action_space.n))

# Policies
def greedy_policy(Q, state):
    return np.argmax(Q[state])

def epsilon_greedy_policy(Q, state, epsilon):
    return greedy_policy(Q, state) if random.random() > epsilon else env.action_space.sample()

# Parameters
n_episodes = 5000
lr, gamma = 0.7, 0.95
max_steps = 100
max_epsilon, min_epsilon, decay_rate = 1.0, 0.1, 0.005

# Training
for episode in range(n_episodes):
    state, _ = env.reset()
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

    for _ in range(max_steps):
        action = epsilon_greedy_policy(Qtable, state, epsilon)
        new_state, reward, terminated, truncated, _ = env.step(action)

        Qtable[state, action] += lr * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state, action])
        state = new_state

        if terminated or truncated:
            break

print(Qtable)


[[  0.           0.           0.           0.           0.
    0.        ]
 [  2.74713848   3.94930404   0.15739588   3.94751888   5.20997639
   -5.05533364]
 [  7.93261639   9.40347558   7.93064278   9.40366768  10.9512375
    0.40365103]
 ...
 [ -2.65639999  12.57918973   5.88977911   5.31295899  -5.20409726
   -5.04521187]
 [ -4.48565518   6.53654881  -4.69955795  -4.51545209 -11.53388534
  -11.61028835]
 [ 14.50861453  -1.620045    -1.82049     18.          -7.
   -9.1       ]]


In [5]:
import imageio
# Evaluation and recording

def evaluate_and_record(Q, filename="taxi_agent.gif", n_episodes=3, max_steps=100):
    frames = []

    for _ in range(n_episodes):
        state, _ = env.reset()

        for _ in range(max_steps):
            frames.append(env.render())
            action = greedy_policy(Q, state)
            state, _, terminated, truncated, _ = env.step(action)

            if terminated or truncated:
                frames.append(env.render())
                break

    imageio.mimsave(filename, frames, fps=2)
    print(f"GIF saved as: {filename}")

evaluate_and_record(Qtable)

GIF saved as: taxi_agent.gif
